# Notebook 01 — Celeb-DF v2 Extraction and Actor-Disjoint Sampling

**Goal:** From the Celeb-DF v2 zip, produce a clean train/val/test split where:
- Every identity appears in exactly one split (no leakage)
- Synthesis (fake) clips are correctly handled — they involve TWO identities (source + target), both must land in the same split or the clip is rejected
- Splits are deterministic across runs (MD5 hash with seed = 42)
- We retain all real clips and as many synthesis clips as possible after the strict actor-disjoint filter

**Inputs:**
- `data/raw_zips/Celeb-DF-v2.zip` — the 9.3 GB official Celeb-DF v2 (6,532 clips)

**Outputs:**
- `data/celeb-df/{Celeb-real,Celeb-synthesis,YouTube-real}/` — extracted videos
- `data/manifest.csv` — full sample list with columns: filename, folder, label, source_id, target_id, actor_ids, split
- `data/splits/{train,val,test}.csv` — per-split manifests
- `data/rejected_clips.csv` — synthesis clips dropped due to cross-split identities (for transparency)

**Filename patterns:**
- Celeb-real: `idN_XXXX.mp4` (N = celebrity identity, XXXX = clip index)
- Celeb-synthesis: `idN_idM_XXXX.mp4` (N = source identity whose face is swapped IN, M = target identity providing the body/scene/audio)
- YouTube-real: `XXXXX.mp4` (each video is a unique person — sequential numbering, no actor grouping)

**Runtime:** ~5-10 minutes (mostly the zip extraction).

## 1. Imports and configuration

In [ ]:
import os
import sys
import re
import json
import shutil
import hashlib
import zipfile
import shutil as _shutil
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm

print(f"Python: {sys.version.split()[0]}")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path('..').resolve()

# Paths
DATA_DIR        = PROJECT_ROOT / 'data'
ZIP_PATH        = DATA_DIR / 'raw_zips' / 'Celeb-DF-v2.zip'
CELEB_DIR       = DATA_DIR / 'celeb-df'              # extracted videos live here
SPLITS_DIR      = DATA_DIR / 'splits'
MANIFEST_CSV    = DATA_DIR / 'manifest.csv'
REJECTED_CSV    = DATA_DIR / 'rejected_clips.csv'

# Split ratios (must sum to 1.0)
SPLIT_RATIOS = {'train': 0.80, 'val': 0.10, 'test': 0.10}

# Reproducibility seed
SAMPLE_SEED = 42

# Whether to delete the zip after extraction (saves 9.3 GB)
DELETE_ZIP_AFTER_EXTRACT = True

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Zip path:     {ZIP_PATH}")
print(f"Extract to:   {CELEB_DIR}")
print(f"Split ratios: {SPLIT_RATIOS}")

## 2. Folder setup

In [ ]:
for d in [DATA_DIR, CELEB_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  OK: {d}")

print("\nFolder structure ready.")

## 3. Extract Celeb-DF v2 zip

Extracts all 6,532 videos to `data/celeb-df/`. Idempotent — if files already exist on disk from a previous run, they're skipped.

After extraction, the zip is deleted to free ~9 GB.

In [ ]:
def get_free_gb(path: Path) -> float:
    """Free disk space at path, in GB."""
    return _shutil.disk_usage(path).free / (1024**3)


def extract_zip_to(zip_path: Path, output_dir: Path) -> dict:
    """Extract every .mp4 from the zip to output_dir, preserving subfolders.
    
    Skips files already on disk. Returns per-folder count of extracted clips.
    """
    counts = Counter()
    
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if m.endswith('.mp4')]
        # Also extract metadata files (e.g. List_of_testing_videos.txt)
        meta_members = [m for m in zf.namelist() if m.endswith('.txt') and not m.endswith('/')]
        
        # Extract metadata first (small, fast)
        for m in meta_members:
            dst = output_dir / m
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                with zf.open(m) as src, open(dst, 'wb') as out:
                    shutil.copyfileobj(src, out)
                print(f"  Extracted metadata: {m}")
        
        # Extract videos with progress bar
        for member in tqdm(members, desc="Extracting videos", unit="clip"):
            dst = output_dir / member
            if dst.exists():
                counts[member.split('/')[0]] += 1
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(member) as src, open(dst, 'wb') as out:
                shutil.copyfileobj(src, out)
            counts[member.split('/')[0]] += 1
    
    return dict(counts)


if not ZIP_PATH.exists():
    print(f"[ERROR] Zip not found at {ZIP_PATH}")
    print("Did you move Celeb-DF-v2.zip to data/raw_zips/?")
    raise FileNotFoundError(str(ZIP_PATH))

print(f"Free disk before extraction: {get_free_gb(DATA_DIR):.1f} GB\n")
extracted_counts = extract_zip_to(ZIP_PATH, CELEB_DIR)
print(f"\nExtraction complete:")
for folder, count in sorted(extracted_counts.items()):
    print(f"  {folder}: {count:,} videos")
print(f"\nFree disk after extraction: {get_free_gb(DATA_DIR):.1f} GB")

In [ ]:
# Optionally delete the zip after successful extraction
if DELETE_ZIP_AFTER_EXTRACT and ZIP_PATH.exists():
    # Sanity check: only delete if we extracted a reasonable number of files
    total_extracted = sum(extracted_counts.values())
    if total_extracted >= 6000:
        ZIP_PATH.unlink()
        print(f"[OK] Deleted zip ({ZIP_PATH.name}). Free disk now: {get_free_gb(DATA_DIR):.1f} GB")
    else:
        print(f"[WARN] Only {total_extracted} videos extracted (expected ~6,500). Keeping zip just in case.")
else:
    print("Zip kept on disk (DELETE_ZIP_AFTER_EXTRACT = False).")

## 4. Filename parser

Decodes each Celeb-DF v2 filename into structured metadata: which identities are involved, REAL vs FAKE, etc.

Critically, `Celeb-synthesis` clips involve TWO identities (source face donor + target body/scene). We track both because both must land in the same split for the actor-disjoint constraint to hold.

In [ ]:
def parse_filename(rel_path: str) -> dict | None:
    """Parse a Celeb-DF v2 relative path into structured metadata.

    Args:
        rel_path: e.g. 'Celeb-real/id0_0000.mp4', 'Celeb-synthesis/id0_id16_0000.mp4'

    Returns dict with keys, or None if the filename doesn't match expected patterns:
        folder      : 'Celeb-real' / 'Celeb-synthesis' / 'YouTube-real'
        filename    : just the basename
        rel_path    : relative path inside celeb-df/
        label       : 'REAL' / 'FAKE'
        source_id   : str (idN for celebs, 'yt_NNNNN' for YouTube clips)
        target_id   : str or None (only for Celeb-synthesis)
        clip_idx    : str (the trailing index, e.g. '0000')
        actor_ids   : frozenset of identities involved
    """
    parts = rel_path.replace('\\', '/').split('/')
    if len(parts) < 2:
        return None
    
    folder = parts[0]
    filename = parts[-1]
    
    if not filename.endswith('.mp4'):
        return None
    name = filename[:-4]  # strip .mp4
    
    if folder == 'Celeb-real':
        m = re.match(r'^(id\d+)_(\d+)$', name)
        if not m: return None
        sid, clip_idx = m.group(1), m.group(2)
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'REAL', 'source_id': sid, 'target_id': None,
            'clip_idx': clip_idx, 'actor_ids': frozenset([sid]),
        }
    
    elif folder == 'Celeb-synthesis':
        m = re.match(r'^(id\d+)_(id\d+)_(\d+)$', name)
        if not m: return None
        sid, tid, clip_idx = m.group(1), m.group(2), m.group(3)
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'FAKE', 'source_id': sid, 'target_id': tid,
            'clip_idx': clip_idx, 'actor_ids': frozenset([sid, tid]),
        }
    
    elif folder == 'YouTube-real':
        m = re.match(r'^(\d+)$', name)
        if not m: return None
        clip_idx = m.group(1)
        # Each YouTube clip is a unique person — its own actor identity
        actor = f'yt_{clip_idx}'
        return {
            'folder': folder, 'filename': filename, 'rel_path': rel_path,
            'label': 'REAL', 'source_id': actor, 'target_id': None,
            'clip_idx': clip_idx, 'actor_ids': frozenset([actor]),
        }
    
    return None


# Sanity check on parser
tests = [
    'Celeb-real/id0_0000.mp4',
    'Celeb-synthesis/id0_id16_0042.mp4',
    'YouTube-real/00050.mp4',
    'Celeb-real/garbage.mp4',  # should return None
]
for t in tests:
    result = parse_filename(t)
    print(f"  {t} -> {result}")

## 5. Actor-to-split hashing

Each unique identity (id0, id1, ..., yt_00050, ...) deterministically hashes to one of train/val/test. Same identity always produces the same split, regardless of which clip we're looking at.

In [ ]:
def actor_to_split(actor_id: str, ratios: dict = SPLIT_RATIOS, seed: int = SAMPLE_SEED) -> str:
    """Map actor_id to a split deterministically via MD5 hash."""
    h = hashlib.md5(f"{seed}_{actor_id}".encode()).hexdigest()
    n = int(h, 16) / (16 ** 32)  # uniform float in [0, 1)
    cum = 0.0
    for split, ratio in ratios.items():
        cum += ratio
        if n < cum:
            return split
    return list(ratios.keys())[-1]


def determine_clip_split(actor_ids: frozenset) -> str | None:
    """Determine which split a clip belongs to, based on ALL its actors.

    Returns 'train', 'val', or 'test' if all actors hash to the same split.
    Returns None if they're split across — caller should reject the clip.
    """
    splits = set(actor_to_split(aid) for aid in actor_ids)
    if len(splits) == 1:
        return splits.pop()
    return None


# Hash distribution sanity check on synthetic identities
test_ids = [f'id{i}' for i in range(100)] + [f'yt_{i:05d}' for i in range(300)]
test_splits = [actor_to_split(a) for a in test_ids]
test_counts = Counter(test_splits)
print("Hash distribution on synthetic IDs (100 celebs + 300 YouTube):")
for split in ['train', 'val', 'test']:
    pct = test_counts[split] / len(test_ids) * 100
    expected = SPLIT_RATIOS[split] * 100
    print(f"  {split}: {test_counts[split]:3d} ({pct:5.1f}% — expected {expected:.0f}%)")

# Determinism check
assert actor_to_split('id0') == actor_to_split('id0'), "Hash not deterministic!"
print("\n[OK] Hashing is deterministic.")

## 6. Build full manifest from extracted files

Walks the extracted `data/celeb-df/` tree, parses every filename, applies actor-disjoint split logic, and records the result. Synthesis clips whose source and target identities hash to different splits are recorded separately in `rejected_clips.csv` — they're the unavoidable cost of strict actor-disjoint protocol.

In [ ]:
rows_kept = []
rows_rejected = []
parse_failures = []

# Walk only the three known subfolders
for subfolder in ['Celeb-real', 'Celeb-synthesis', 'YouTube-real']:
    folder_path = CELEB_DIR / subfolder
    if not folder_path.exists():
        print(f"[WARN] Folder missing: {folder_path}")
        continue
    
    mp4_files = sorted(folder_path.glob('*.mp4'))
    print(f"\n{subfolder}: {len(mp4_files):,} videos found")
    
    for f in mp4_files:
        rel_path = f"{subfolder}/{f.name}"
        meta = parse_filename(rel_path)
        
        if meta is None:
            parse_failures.append(rel_path)
            continue
        
        split = determine_clip_split(meta['actor_ids'])
        meta['split'] = split  # may be None
        meta['actor_ids_str'] = '|'.join(sorted(meta['actor_ids']))  # for CSV storage
        
        if split is None:
            # Multi-identity clip whose identities span different splits — reject
            rows_rejected.append(meta)
        else:
            rows_kept.append(meta)

print(f"\n--- Build complete ---")
print(f"  Clips kept:       {len(rows_kept):,}")
print(f"  Clips rejected:   {len(rows_rejected):,} (synthesis clips whose source+target span splits)")
print(f"  Parse failures:   {len(parse_failures):,}")
if parse_failures:
    print(f"  Example failures: {parse_failures[:5]}")

## 7. Save manifests

In [ ]:
# Columns to save (omit the frozenset, keep the string version)
cols = ['rel_path', 'folder', 'filename', 'label', 'source_id', 'target_id',
        'clip_idx', 'actor_ids_str', 'split']

manifest_df = pd.DataFrame(rows_kept)[cols]
manifest_df.to_csv(MANIFEST_CSV, index=False)
print(f"Wrote {MANIFEST_CSV} ({len(manifest_df):,} rows)")

if rows_rejected:
    rejected_df = pd.DataFrame(rows_rejected)[cols[:-1]]  # split is always None for rejected
    rejected_df.to_csv(REJECTED_CSV, index=False)
    print(f"Wrote {REJECTED_CSV} ({len(rejected_df):,} rows)")

# Per-split CSVs
for split in ['train', 'val', 'test']:
    split_df = manifest_df[manifest_df['split'] == split].copy()
    out_path = SPLITS_DIR / f'{split}.csv'
    split_df.to_csv(out_path, index=False)
    print(f"  {split}: {len(split_df):,} clips → {out_path}")

## 8. Verification — split sizes, balance, leakage check

Critical sanity checks. Failure here means the sampling logic has a bug — do not proceed to preprocessing.

In [ ]:
manifest_df = pd.read_csv(MANIFEST_CSV)

print("=" * 60)
print("MANIFEST OVERVIEW")
print("=" * 60)
print(f"Total clips kept:         {len(manifest_df):,}")
print(f"Unique identities seen:   {manifest_df['actor_ids_str'].str.split('|').explode().nunique():,}")

# --- Per-split, per-class breakdown ---
print("\n--- Split sizes by label ---")
split_label = manifest_df.groupby(['split', 'label']).size().unstack(fill_value=0)
split_label['TOTAL'] = split_label.sum(axis=1)
print(split_label)

# --- Per-split, per-source-folder breakdown ---
print("\n--- Split sizes by source folder ---")
split_folder = manifest_df.groupby(['split', 'folder']).size().unstack(fill_value=0)
print(split_folder)

# --- Class balance ---
print("\n--- Class balance per split ---")
for split in ['train', 'val', 'test']:
    sub = manifest_df[manifest_df['split'] == split]
    if len(sub) == 0:
        continue
    n_real = (sub['label'] == 'REAL').sum()
    n_fake = (sub['label'] == 'FAKE').sum()
    print(f"  {split}: REAL={n_real:,} ({n_real/len(sub)*100:.1f}%), FAKE={n_fake:,} ({n_fake/len(sub)*100:.1f}%)")

# --- Leakage check ---
print("\n--- Actor-disjoint leakage check ---")
# Build identity → set-of-splits mapping by exploding the pipe-separated actor_ids_str
exploded = manifest_df.assign(
    actor=manifest_df['actor_ids_str'].str.split('|')
).explode('actor')
actor_splits = exploded.groupby('actor')['split'].nunique()
leaked = actor_splits[actor_splits > 1]
if len(leaked) == 0:
    print(f"  [OK] Every one of {len(actor_splits):,} unique identities appears in exactly one split.")
else:
    print(f"  [FAIL] {len(leaked)} identities appear in multiple splits!")
    print(leaked.head(10))
    raise AssertionError("Actor-disjoint constraint violated")

# --- Rejection rate ---
if REJECTED_CSV.exists():
    rej = pd.read_csv(REJECTED_CSV)
    total_synthesis_attempted = len(manifest_df[manifest_df['folder']=='Celeb-synthesis']) + len(rej)
    rejection_pct = len(rej) / total_synthesis_attempted * 100
    print(f"\n--- Rejected synthesis clips ---")
    print(f"  Rejected: {len(rej):,} / {total_synthesis_attempted:,} synthesis clips ({rejection_pct:.1f}%)")
    print("  These are synthesis clips where source and target identities hash to different splits.")
    print("  Cost of strict actor-disjoint protocol — not a bug.")

## 9. Disk usage summary

In [ ]:
def folder_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return sum(f.stat().st_size for f in path.rglob('*') if f.is_file()) / (1024**3)

print("Disk usage:")
print(f"  data/raw_zips/     : {folder_size_gb(DATA_DIR / 'raw_zips'):.2f} GB")
print(f"  data/celeb-df/     : {folder_size_gb(CELEB_DIR):.2f} GB")
print(f"  data/splits/       : {folder_size_gb(SPLITS_DIR):.4f} GB")
print(f"\n  Free disk:         : {get_free_gb(DATA_DIR):.1f} GB")

## 10. Next steps

If everything looks good:
1. **Update `project_log.md`** — paste the verification cell output under Section 1 (Dataset prep log). Include split sizes, class balance per split, total identities, and rejection count.
2. **Commit + push** the updated notebook and project log to the GitHub repo.
3. **Move on to notebook 02** (`02_preprocessing.ipynb`) — MTCNN face detection on every clip in `data/celeb-df/`, audio extraction at 16 kHz, save 16-frame face crop sequences and 10-second audio clips.

If anything in the verification looks off (zero clips in a split, leakage detected, balance way off), stop and debug here.

**Expected ballpark numbers** (your actual values will be close to these):
- Total clips kept: ~5,500-6,000 (out of 6,532 — rejection rate roughly 5-10%)
- Train: ~80% of kept | Val: ~10% | Test: ~10%
- Class balance: heavily fake-leaning (~85% FAKE, ~15% REAL) — this is Celeb-DF's natural distribution; we report AUC which handles this gracefully